# 📥 TMS2 - Setup Kaggle Datasets (Manual Upload)

**Instructions:**
1. Download datasets manually from Kaggle (links below)
2. Upload ZIP files to `tms2_colab_training/data/kaggle/` on Google Drive
3. Run this notebook to unzip and verify

### Dataset Links:
| Dataset | Download Link | Folder Name |
|---------|--------------|-------------|
| ⭐ UA-DETRAC | [Download](https://www.kaggle.com/datasets/dtrnngc/ua-detrac-dataset) | `ua-detrac/` |
| Real-Time Traffic | [Download](https://www.kaggle.com/datasets/unidpro/real-time-traffic-video-dataset) | `real-time-traffic/` |
| Singapore Density | [Download](https://www.kaggle.com/datasets/rahat52/traffic-density-singapore) | `traffic-density-singapore/` |
| Serbia Traffic | [Download](https://www.kaggle.com/datasets/unidpro/road-traffic-in-serbia-videos-and-images) | `serbia-traffic/` |
| LISA Traffic Light | [Download](https://www.kaggle.com/datasets/mbornoe/lisa-traffic-light-dataset) | `lisa-traffic-light/` |

---

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/tms2_colab_training'
DATA_PATH = f'{DRIVE_PATH}/data'
KAGGLE_PATH = f'{DATA_PATH}/kaggle'

import os
os.makedirs(KAGGLE_PATH, exist_ok=True)

print(f"Upload your ZIP files to: {KAGGLE_PATH}")

## 2. Check What's Uploaded

In [ ]:
import glob

print("📂 Files in kaggle folder:")
!ls -la {KAGGLE_PATH}/

# Find ZIP files
zip_files = glob.glob(f'{KAGGLE_PATH}/*.zip')
print(f"\n📦 Found {len(zip_files)} ZIP files to extract:")
for zf in zip_files:
    size_mb = os.path.getsize(zf) / (1024**2)
    print(f"   - {os.path.basename(zf)} ({size_mb:.1f} MB)")

## 3. Unzip All Datasets

In [ ]:
import zipfile
import shutil

# Expected dataset names (maps ZIP name patterns to folder names)
DATASET_FOLDERS = {
    'ua-detrac': 'ua-detrac',
    'detrac': 'ua-detrac',
    'real-time-traffic': 'real-time-traffic',
    'traffic-density-singapore': 'traffic-density-singapore',
    'singapore': 'traffic-density-singapore',
    'road-traffic-in-serbia': 'serbia-traffic',
    'serbia': 'serbia-traffic',
    'lisa-traffic-light': 'lisa-traffic-light',
    'lisa': 'lisa-traffic-light',
}

def get_folder_name(zip_name):
    """Determine output folder from ZIP name."""
    zip_lower = zip_name.lower()
    for pattern, folder in DATASET_FOLDERS.items():
        if pattern in zip_lower:
            return folder
    # Default: use zip name without extension
    return zip_name.replace('.zip', '').replace('-dataset', '')

# Unzip each file
for zip_path in zip_files:
    zip_name = os.path.basename(zip_path)
    folder_name = get_folder_name(zip_name)
    extract_path = f'{KAGGLE_PATH}/{folder_name}'
    
    print(f"\n📦 Extracting: {zip_name}")
    print(f"   → {extract_path}")
    
    try:
        # Create folder
        os.makedirs(extract_path, exist_ok=True)
        
        # Extract
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(extract_path)
        
        # Count files
        all_files = glob.glob(f'{extract_path}/**/*', recursive=True)
        file_count = len([f for f in all_files if os.path.isfile(f)])
        print(f"   ✅ Extracted {file_count} files")
        
        # Optional: delete ZIP to save space
        # os.remove(zip_path)
        # print(f"   🗑️ Deleted ZIP")
        
    except Exception as e:
        print(f"   ❌ Error: {e}")

print("\n✅ Extraction complete!")

## 4. Verify All Datasets

In [ ]:
print("\n" + "="*60)
print("📊 DATASET SUMMARY")
print("="*60)

expected_datasets = {
    'ua-detrac': '⭐ UA-DETRAC (YOLOv8 Benchmark)',
    'real-time-traffic': 'Real-Time Traffic (YOLOv8)',
    'traffic-density-singapore': 'Traffic Density Singapore (LSTM)',
    'serbia-traffic': 'Serbia Traffic (YOLOv8)',
    'lisa-traffic-light': 'LISA Traffic Light (RL)',
    'highway-traffic-videos': 'Highway Traffic (existing)',
}

total_files = 0
total_size = 0

for folder, name in expected_datasets.items():
    path = f'{KAGGLE_PATH}/{folder}'
    if os.path.exists(path):
        files = glob.glob(f'{path}/**/*', recursive=True)
        files = [f for f in files if os.path.isfile(f)]
        size = sum(os.path.getsize(f) for f in files) / (1024**3)
        total_files += len(files)
        total_size += size
        print(f"✅ {name}: {len(files)} files ({size:.2f} GB)")
    else:
        print(f"❌ {name}: Not found")

print(f"\n📁 Total: {total_files} files ({total_size:.2f} GB)")

In [ ]:
# Show folder structure
print("\n📂 Folder Structure:")
!ls -la {KAGGLE_PATH}/

## ✅ Done!

Your datasets are ready. Now proceed with:

1. **`01_Data_Generation.ipynb`** - Process videos → LSTM data
2. **`02_LSTM_Training.ipynb`** - Train prediction models
3. **`03_RL_Training.ipynb`** - Train signal controller
4. **`04_YOLOv8_Finetuning.ipynb`** - Fine-tune detection